<a href="https://colab.research.google.com/github/bikebhunia/My_YOLO/blob/main/YOLOv8_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install ultralytics
!pip install filterpy
!pip install timm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 29.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.0/178.0 kB 10.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for filterpy: filename=filterpy-1.4.5-py3-none-any.whl size=110460 sha256=0c31f83a1dd03f4b4cca6a2c814e1611db769115e7400f9f4049d6e67df5d82c
  Stored in directory: /root/.cache/pip/wheels/77/bf/4c/b0c3f4798a0166668752312a67118b27a3cd341e13ac0ae6ee
Successfully built filterpy


In [ ]:
from ultralytics import YOLO
import cv2
from google.colab.patches import cv2_imshow
from google.colab import files
import matplotlib.pyplot as plt
import numpy as np
import torch
from filterpy.kalman import KalmanFilter
from scipy.optimize import linear_sum_assignment
import os
from collections import defaultdict

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [ ]:
VIDEO_PATH = '/content/VIDEO-2026-01-23-18-33-43.mp4'  # Your video file
OUTPUT_PATH = '/content/output_tracked_VIDEO-2026-01-23-18-33-43.mp4'  # Output file
MODEL_PATH = 'yolov8s.pt'  # Change to 'best.pt' if using custom trained model


In [ ]:
# Camera calibration (adjust based on your camera setup)
FOCAL_LENGTH = 1000  # pixels (adjust for your camera)
KNOWN_WIDTH = 1.8  # Average vehicle width in meters
FRAME_HISTORY = 30  # Number of frames to store for trajectory prediction

# Vehicle classes in COCO dataset
VEHICLE_CLASSES = {
    2: 'car',
    3: 'motorcycle',
    5: 'bus',
    7: 'truck'
}

In [ ]:
def calculate_distance(bbox_width, known_width=KNOWN_WIDTH, focal_length=FOCAL_LENGTH):
    """
    Calculate distance using perspective geometry
    Distance = (Known_Width × Focal_Length) / Perceived_Width
    """
    if bbox_width > 0:
        distance = (known_width * focal_length) / bbox_width
        return distance
    return 0
def predict_trajectory(track_history, num_future_points=10):
    """
    Predict future trajectory using linear extrapolation
    """
    if len(track_history) < 2:
        return []


  # Get last few points for velocity calculation
    recent_points = track_history[-min(5, len(track_history)):]

    # Calculate average velocity
    velocities = []
    for i in range(1, len(recent_points)):
        vx = recent_points[i][0] - recent_points[i-1][0]
        vy = recent_points[i][1] - recent_points[i-1][1]
        velocities.append((vx, vy))

    if not velocities:
        return []

    # Average velocity
    avg_vx = np.mean([v[0] for v in velocities])
    avg_vy = np.mean([v[1] for v in velocities])


    # Predict future points
    last_point = track_history[-1]
    predicted_points = []

    for i in range(1, num_future_points + 1):
        pred_x = int(last_point[0] + avg_vx * i)
        pred_y = int(last_point[1] + avg_vy * i)
        predicted_points.append((pred_x, pred_y))

    return predicted_points

In [ ]:
def draw_tracking_info(frame, bbox, track_id, class_name, distance, color=(0, 255, 0)):
    """
    Draw bounding box, track ID, and distance information
    """
    x1, y1, x2, y2 = map(int, bbox)

    # Draw bounding box
    cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)

    # Prepare label with track ID, class, and distance
    label = f"ID:{track_id} {class_name} {distance:.1f}m"

    # Calculate label size and position
    (label_w, label_h), baseline = cv2.getTextSize(
        label, cv2.FONT_HERSHEY_SIMPLEX, 0.6, 2
    )

    # Draw label background
    cv2.rectangle(
        frame,
        (x1, y1 - label_h - baseline - 5),
        (x1 + label_w, y1),
        color,
        -1
    )

    # Draw label text
    cv2.putText(
        frame,
        label,
        (x1, y1 - baseline - 5),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.6,
        (0, 0, 0),
        2
    )

In [ ]:
def draw_trajectory(frame, track_history, predicted_points, color=(0, 255, 0)):
    """
    Draw historical path (green line) and predicted trajectory (blue dots)
    """
    # Draw historical path
    if len(track_history) > 1:
        points = np.array(track_history, dtype=np.int32).reshape((-1, 1, 2))
        cv2.polylines(frame, [points], isClosed=False, color=color, thickness=2)

    # Draw predicted trajectory as blue dots
    for point in predicted_points:
        if 0 <= point[0] < frame.shape[1] and 0 <= point[1] < frame.shape[0]:
            cv2.circle(frame, point, 4, (255, 0, 0), -1)  # Blue dots

In [ ]:
def draw_stats(frame, frame_count, active_tracks):
    """
    Draw frame count and active tracks on frame
    """
    # Create semi-transparent overlay for stats
    overlay = frame.copy()
    cv2.rectangle(overlay, (10, 10), (300, 80), (0, 0, 0), -1)
    cv2.addWeighted(overlay, 0.6, frame, 0.4, 0, frame)

    # Draw stats text
    cv2.putText(
        frame,
        f"Frame: {frame_count}",
        (20, 35),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7,
        (255, 255, 255),
        2
    )

    cv2.putText(
        frame,
        f"Active Tracks: {active_tracks}",
        (20, 65),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7,
        (255, 255, 255),
        2
    )

In [ ]:
def track_vehicles(video_path, output_path, model_path=MODEL_PATH):
    """
    Main function to track vehicles with all visualization features
    """
    # Load YOLO model
    print("Loading YOLO model...")
    model = YOLO(model_path)

    # Open video
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"Error: Cannot open video file {video_path}")
        return

    # Get video properties
    fps = int(cap.get(cv2.CAP_PROP_FPS))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    print(f"Video Info: {width}x{height} @ {fps}fps, Total frames: {total_frames}")

    # Initialize video writer
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

    # Initialize tracking storage
    track_history = defaultdict(lambda: [])
    frame_count = 0

    print("Processing video...")

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        frame_count += 1

        # Run YOLO tracking (persist=True maintains track IDs)
        results = model.track(
            frame,
            persist=True,
            classes=list(VEHICLE_CLASSES.keys()),
            verbose=False,
            tracker="bytetrack.yaml"  # Use ByteTrack for better tracking
        )

        # Process detections
        active_tracks = 0

        if results[0].boxes is not None and results[0].boxes.id is not None:
            boxes = results[0].boxes.xyxy.cpu().numpy()
            track_ids = results[0].boxes.id.cpu().numpy().astype(int)
            classes = results[0].boxes.cls.cpu().numpy().astype(int)
            confidences = results[0].boxes.conf.cpu().numpy()

            active_tracks = len(track_ids)

            for box, track_id, cls, conf in zip(boxes, track_ids, classes, confidences):
                if conf < 0.5:  # Skip low confidence detections
                    continue

                x1, y1, x2, y2 = box

                # Calculate center point
                center_x = int((x1 + x2) / 2)
                center_y = int((y1 + y2) / 2)

                # Store track history
                track_history[track_id].append((center_x, center_y))

                # Keep only recent history
                if len(track_history[track_id]) > FRAME_HISTORY:
                    track_history[track_id].pop(0)

                # Calculate distance
                bbox_width = x2 - x1
                distance = calculate_distance(bbox_width)

                # Get class name
                class_name = VEHICLE_CLASSES.get(cls, 'vehicle')

                # Generate color based on track ID (green tones)
                np.random.seed(track_id)
                color = (0, np.random.randint(180, 255), np.random.randint(100, 200))

                # Draw bounding box with info
                draw_tracking_info(frame, box, track_id, class_name, distance, color)

                # Predict trajectory
                predicted_points = predict_trajectory(track_history[track_id])

                # Draw trajectory
                draw_trajectory(frame, track_history[track_id], predicted_points, color)

                # Draw stats
        draw_stats(frame, frame_count, active_tracks)


        # Write frame
        out.write(frame)

        # Progress update
        if frame_count % 30 == 0:
            progress = (frame_count / total_frames) * 100
            print(f"Progress: {progress:.1f}% ({frame_count}/{total_frames} frames)")

    # Cleanup
    cap.release()
    out.release()

    print(f"\n✓ Processing complete!")
    print(f"✓ Output saved to: {output_path}")
    print(f"✓ Total frames processed: {frame_count}")
    print(f"✓ Total unique tracks: {len(track_history)}")

    return output_path

In [ ]:
if __name__ == "__main__":
    print("="*70)
    print("VEHICLE TRACKING SYSTEM")
    print("="*70)
    print(f"Input Video:  {VIDEO_PATH}")
    print(f"Output Video: {OUTPUT_PATH}")
    print(f"Model:        {MODEL_PATH}")
    print("="*70 + "\n")

    # Run tracking
    output_file = track_vehicles(VIDEO_PATH, OUTPUT_PATH, MODEL_PATH)

    print("\n" + "="*70)
    print("TRACKING COMPLETE!")
    print("="*70)
    print("\nFeatures in output video:")
    print("✓ Green bounding boxes around vehicles")
    print("✓ Track IDs for each vehicle")
    print("✓ Distance estimates in meters")
    print("✓ Blue dots showing predicted trajectory")
    print("✓ Frame count and active tracks display")
    print("\nDownload your output video from Colab files panel!")
    print("="*70)

VEHICLE TRACKING SYSTEM
Input Video:  /content/VIDEO-2026-01-23-18-33-43.mp4
Output Video: /content/output_tracked_VIDEO-2026-01-23-18-33-43.mp4
Model:        yolov8s.pt

Loading YOLO model...
Video Info: 480x272 @ 24fps, Total frames: 720
Processing video...
requirements: Ultralytics requirement ['lap>=0.5.12'] not found, attempting AutoUpdate...
Using Python 3.12.12 environment at: /usr
Resolved 2 packages in 105ms
Prepared 1 package in 24ms
Installed 1 package in 2ms
 + lap==0.5.12

requirements: AutoUpdate success ✅ 0.6s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect

Progress: 4.2% (30/720 frames)
Progress: 8.3% (60/720 frames)
Progress: 12.5% (90/720 frames)
Progress: 16.7% (120/720 frames)
Progress: 20.8% (150/720 frames)
Progress: 25.0% (180/720 frames)
Progress: 29.2% (210/720 frames)
Progress: 33.3% (240/720 frames)
Progress: 37.5% (270/720 frames)
Progress: 41.7% (300/720 frames)
Progress: 45.8% (330/720 frames)
Progress: 50.0% (360/720 